# Versuch v7: Quellen mit Zitatprüfung

Ziel: Falsche Quellenangaben automatisch erkennen, statt sie von Hand zu suchen.

- Jeder Bericht bekommt eine Kennung (`B1`, `B2`, ...). Das Modell schreibt keine Quellenliste mehr, sondern belegt **jeden Satz** mit Kennung und kurzem **wörtlichem Zitat**.
- Das Notebook prüft, ob das Zitat wirklich im genannten Bericht steht. Nur geprüfte Belege werden als Quelle angezeigt; **Titel und Datum kommen aus den Berichten selbst**, nicht vom Modell.
- Aufbau sonst wie v5/v6: alle Berichte vollständig im Prompt, keine Suche, mehrere Durchläufe.
- **Grenze:** Ein gefundenes Zitat zeigt, dass der Bericht den Satz enthält, nicht dass die Aussage der Antwort daraus folgt.

In [ ]:
!pip install -q openai
!git clone -q https://github.com/Julien0049/AI-database.git || git -C AI-database pull -q

In [ ]:
from google.colab import userdata
from openai import OpenAI

KEY_NAME = "OPENROUTER_API_KEY"
MODEL = "google/gemini-2.5-pro"

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get(KEY_NAME))

## Berichte mit Kennung und Metadaten

Titel und Datum werden aus Überschrift und `**Datum:**`-Zeile gelesen.

In [ ]:
import pathlib, re

DATA = pathlib.Path("AI-database/data/synthetic_reports")

reports = {}
for n, path in enumerate(sorted(DATA.glob("AAR-*.md")), 1):
    text = path.read_text()
    title = re.search(r"^# (?:After-Action-Report: )?(.+)$", text, re.M).group(1).strip()
    date = re.search(r"\*\*Datum:\*\* (\S+)", text).group(1)
    reports[f"B{n}"] = {"datei": path.name, "titel": title, "datum": date, "text": text}

corpus = "\n\n".join(f'<bericht id="{rid}">\n{r["text"]}\n</bericht>' for rid, r in reports.items())
questions = re.findall(r"\*\*Frage\*\*\s*\n\s*> (.+)", (DATA / "test_questions.md").read_text())
print(len(reports), "Berichte,", len(questions), "Fragen")
for rid, r in reports.items():
    print(rid, r["datum"], r["titel"])

## Antwortregeln und Ausgabeformat

Regeln 1–6 wie in v5/v6. Neu ist nur das Format: JSON mit Sätzen und Belegen.

In [ ]:
SYSTEM = """Du beantwortest Fragen ausschließlich auf Grundlage der mitgelieferten Berichte.
Die Berichte sind fiktive Testdaten. Jeder Bericht hat eine Kennung wie B1, B2.

Regeln:
1. Verwende nur Aussagen aus den Berichten. Ergänze nichts aus eigenem Wissen.
2. Ergänzende Aussagen aus mehreren Berichten darfst du zu einer klaren Antwort zusammenführen.
3. Bei Regeln und Verfahren: Wenn Berichte mit vergleichbarem Kontext sich widersprechen, bestimmt die jüngste passende Aussage die Antwort.
   Nenne ältere abweichende Verfahren nicht als gleichrangige Alternativen.
   Bei Erfahrungen und Einschätzungen (zum Beispiel, was sich bewährt hat): Gib unterschiedliche Sichtweisen nebeneinander wieder
   und weise darauf hin, dass die Berichte auseinandergehen. Das Datum allein begründet hier keine Priorität.
4. Wenn Berichte unterschiedliche Kontexte betreffen oder keine sichere Priorisierung zulassen, behaupte keine Priorität.
5. Bezeichne keine Quelle als gültig oder verbindlich; die Berichte belegen das nicht.
6. Wenn die Berichte die Frage nicht ausreichend beantworten, sag das offen und fülle keine Lücken.
   Das gilt auch für jede Teilfrage: Wenn zu einem Teil der Frage keine Informationen vorliegen, schreibe ausdrücklich,
   dass die Berichte dazu nichts sagen. Gib dann nicht stattdessen eine verwandte Aussage als Antwort aus.

Format: Antworte nur mit JSON in dieser Form:
{"saetze": [{"text": "Ein Satz der Antwort.", "belege": [{"bericht": "B3", "zitat": "wörtliche Stelle aus B3"}]}]}
- Jeder Satz, der eine Aussage aus den Berichten enthält, bekommt mindestens einen Beleg.
- Das Zitat ist eine kurze, wörtlich kopierte Stelle aus genau dem genannten Bericht (einige Wörter bis ein Satz).
- Nenne nur Berichte, aus denen die Aussage des Satzes stammt.
- Sätze, die eine Lücke benennen (die Berichte sagen dazu nichts), haben eine leere Beleg-Liste.
- Nenne in den Sätzen keine Titel, Daten oder Kennungen; die Quellen werden automatisch ergänzt."""

## Zitatprüfung

Vergleicht Zitate unabhängig von Groß-/Kleinschreibung, Leerzeichen, Anführungszeichen und Satzzeichen am Rand.

- **gültig:** Zitat steht im genannten Bericht.
- **falscher Bericht:** Zitat steht in einem anderen Bericht.
- **nicht gefunden:** Zitat steht in keinem Bericht (umformuliert oder erfunden).

In [ ]:
import json

def norm(s):
    s = s.lower()
    for a, b in [("„", '"'), ("“", '"'), ("”", '"'), ("‚", "'"), ("‘", "'"), ("’", "'"), ("–", "-"), ("—", "-")]:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip(" .,;:\"'")

def parse(raw):
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip())
    return json.loads(raw)

def check(answer):
    for satz in answer["saetze"]:
        for beleg in satz.get("belege", []):
            zitat = norm(beleg.get("zitat", ""))
            rid = beleg.get("bericht")
            if rid in reports and zitat and zitat in norm(reports[rid]["text"]):
                beleg["status"] = "gültig"
            elif zitat and any(zitat in norm(r["text"]) for r in reports.values()):
                beleg["status"] = "falscher Bericht"
            else:
                beleg["status"] = "nicht gefunden"
    return answer

def render(answer):
    text = " ".join(s["text"] for s in answer["saetze"])
    valid = sorted({b["bericht"] for s in answer["saetze"] for b in s.get("belege", []) if b["status"] == "gültig"},
                   key=lambda rid: int(rid[1:]))
    problems = [f'{b["bericht"]}: {b["status"]} – "{b.get("zitat", "")}"'
                for s in answer["saetze"] for b in s.get("belege", []) if b["status"] != "gültig"]
    out = text
    if valid:
        out += "\n\nQuellen:\n" + "\n".join(f'- „{reports[r]["titel"]}“ ({reports[r]["datum"]})' for r in valid)
    if problems:
        out += "\n\nNicht bestätigte Belege:\n" + "\n".join(f"- {p}" for p in problems)
    return out

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
import openai

RUNS = 3      # Anzahl Durchläufe; die Antworten schwanken zwischen Läufen
PARALLEL = 5  # Anzahl gleichzeitiger Anfragen

def ask(question, retries=5):
    """Gibt (Antwort, Fehler) zurück; wartet bei Überlastung und versucht es erneut."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": f"{corpus}\n\nFrage: {question}"}
                ]
            )
            return response.choices[0].message.content, None
        except (openai.RateLimitError, openai.APIConnectionError, openai.InternalServerError) as e:
            if attempt == retries - 1:
                return None, str(e)
            time.sleep(5 * 2 ** attempt)

jobs = [(d, i, q) for d in range(1, RUNS + 1) for i, q in enumerate(questions, 1)]
start = time.time()
with ThreadPoolExecutor(max_workers=PARALLEL) as pool:
    answers = list(pool.map(lambda job: ask(job[2]), jobs))
DURATION = round(time.time() - start)
print(f"{len(jobs)} Anfragen in {DURATION} s\n")

results = []
for (durchlauf, i, question), (raw, error) in zip(jobs, answers):
    entry = {"durchlauf": durchlauf, "nr": i, "frage": question, "roh": raw}
    if error:
        entry["fehler"] = f"Anfrage fehlgeschlagen: {error}"
        shown = entry["fehler"]
    else:
        try:
            entry["antwort"] = check(parse(raw))
            shown = render(entry["antwort"])
        except (json.JSONDecodeError, KeyError, TypeError) as e:
            entry["fehler"] = f"Ausgabe nicht auswertbar: {e}"
            shown = entry["fehler"] + "\n" + raw
    results.append(entry)
    print(f"## Durchlauf {durchlauf}, Frage {i}\n{question}\n\n{shown}\n")

## Kennzahlen

In [ ]:
from collections import Counter

status = Counter(b["status"] for e in results if "antwort" in e
                 for s in e["antwort"]["saetze"] for b in s.get("belege", []))
unbelegt = sum(1 for e in results if "antwort" in e for s in e["antwort"]["saetze"] if not s.get("belege"))
summary = {
    "antworten": len(results),
    "nicht_auswertbar": sum("fehler" in e for e in results),
    "belege": dict(status),
    "anteil_gueltig": round(status["gültig"] / max(sum(status.values()), 1), 3),
    "saetze_ohne_beleg": unbelegt,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

## Ergebnisse speichern

Sätze ohne Beleg sind nicht automatisch Fehler: Lückenhinweise haben bewusst keinen Beleg. Sie werden bei der Auswertung von Hand geprüft.

In [ ]:
import datetime
from google.colab import files

run = {
    "versuch": "v7 Zitatprüfung",
    "modell": MODEL,
    "zeitpunkt": datetime.datetime.now().isoformat(timespec="seconds"),
    "durchlaeufe": RUNS,
    "dauer_s": DURATION,
    "systemprompt": SYSTEM,
    "berichte": {rid: {k: r[k] for k in ("datei", "titel", "datum")} for rid, r in reports.items()},
    "kennzahlen": summary,
    "ergebnisse": results,
}
name = f"ergebnisse_{MODEL.replace('/', '_')}_{datetime.date.today()}_v7.json"
pathlib.Path(name).write_text(json.dumps(run, ensure_ascii=False, indent=2))
files.download(name)